# Feature engineering
**Date :** 25 au 29/10/2026

Une partie par source : délinquance, LOVAC, équipements, parc de logements, Filosofi, DVF+ (volume), taux des crédits.
Même déroulé pour chacune : préparation à la commune (sections 0 à 5), puis section 6 : rattachement aux sous-bassins par `liaison_bv_scindes`, agrégation, **jointure à gauche sur `cible_sbv_annee`**, contrôle du nombre de lignes, export `fe_cible_<source>_sbv.parquet`.

Le code des sections 6 est identique à celui de `feature_engineering.py` (mêmes fonctions, mêmes fichiers exportés) : toute modification se fait dans les deux.

## Imports, tables communes et fonctions

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# config.py se trouve à la racine du projet, dossier parent du notebook
sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_OUT

# Tables de référence communes à toutes les sources
cible = pd.read_parquet(DATA_OUT / "cible_sbv_annee.parquet")
liaison = pd.read_parquet(DATA_OUT / "liaison_bv_scindes.parquet")

In [2]:
def taux_lisses(table, maille, familles, fenetre=3):
    """Taux annuels (taux1_) et lissés sur `fenetre` années (taux3_) par maille x année,
    pour 1 000 habitants (1 000 logements pour les cambriolages).
    Faits et dénominateurs sont additionnés avant le calcul des taux.
    Le taux lissé reste vide tant que la fenêtre n'est pas complète (premières années)."""
    table = table.copy()
    table["estimes_x_pop"] = table["part_indicateurs_estimes"] * table["insee_pop"]
    colonnes = ["nb_" + f for f in familles] + ["insee_pop", "insee_log", "estimes_x_pop"]
    # 1. Faits et dénominateurs additionnés par maille et par année
    agr = table.groupby([maille, "annee"])[colonnes].sum().reset_index()
    agr = agr.sort_values([maille, "annee"]).reset_index(drop=True)
    # 2. Fiabilité : moyenne pondérée par la population
    agr["part_indicateurs_estimes"] = agr["estimes_x_pop"] / agr["insee_pop"]
    # 3. Taux annuel et taux lissé (années passées uniquement, fenêtre complète)
    for f in familles:
        denom = "insee_log" if f == "cambriolages" else "insee_pop"
        faits_n = agr.groupby(maille)["nb_" + f].rolling(fenetre, min_periods=fenetre).sum().reset_index(level=0, drop=True)
        denom_n = agr.groupby(maille)[denom].rolling(fenetre, min_periods=fenetre).sum().reset_index(level=0, drop=True)
        agr["taux1_" + f] = agr["nb_" + f] / agr[denom] * 1000
        agr["taux3_" + f] = faits_n / denom_n * 1000
    return agr.drop(columns="estimes_x_pop")


def indicateurs_logement(e):
    """Calcule les 12 indicateurs à partir d'effectifs additionnés à une maille
    (mêmes formules que le bloc 7 de la préparation)."""
    e = e.copy()
    for d in ["P22_LOG", "P22_RP", "P22_MEN"]:
        e[d] = e[d].where(e[d] > 0)   # dénominateur nul -> valeur vide au lieu d'un infini

    r = pd.DataFrame(index=e.index)
    r["taux_vacance"] = e["P22_LOGVAC"] / e["P22_LOG"] * 100
    r["part_rsec"] = e["P22_RSECOCC"] / e["P22_LOG"] * 100
    r["part_apparts"] = e["P22_APPART"] / e["P22_LOG"] * 100
    r["log_nb_logements"] = np.log10(e["P22_LOG"])
    r["part_moins40m2"] = (e["P22_RP_M30M2"] + e["P22_RP_3040M2"]) / e["P22_RP"] * 100
    r["part_avant1946"] = (e["P22_RP_ACH1919"] + e["P22_RP_ACH1945"]) / e["P22_RP"] * 100
    r["part_2006_2019"] = e["P22_RP_ACH2019"] / e["P22_RP"] * 100
    r["anciennete_moyenne"] = e["P22_ANEM_RP"] / e["P22_RP"]
    r["taille_menage"] = e["P22_NPER_RP"] / e["P22_RP"]
    r["part_sans_voiture"] = 100 - e["P22_RP_VOIT1P"] / e["P22_MEN"] * 100
    r["part_parking"] = e["P22_RP_GARL"] / e["P22_MEN"] * 100
    r["part_suroccupes"] = (e["C22_RP_SUROCC_MOD"] + e["C22_RP_SUROCC_ACC"]) / e["P22_RP"] * 100
    return r


def joindre_cible(table_sbv, cles, nom_fichier):
    """Jointure à gauche sur la cible, contrôle du nombre de lignes, export."""
    final = pd.merge(cible, table_sbv, on=cles, how="left", validate="many_to_one")
    assert len(final) == len(cible), f"Nombre de lignes modifié par la jointure : {nom_fichier}"
    final.to_parquet(DATA_OUT / nom_fichier, index=False)
    print("-> Exporté :", nom_fichier, final.shape)
    return final

# Feature engineering – Délinquance
**Contributeurs :** Cécile & Théophile · **Date :** 26/10/2026, mis à jour le 29/10/2026

Préparation à la commune × année ; agrégation au sous-bassin × année à la section 6. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA).

## 0. Chargement

In [3]:
CLES = ["code_commune", "annee"]    # clés de jointure avec la table cible

df = pd.read_parquet(DATA_OUT / "delinquance_commune_annee.parquet")

# Contrôle : une seule ligne par commune et par année
print(df.shape)
print("Doublons sur les clés :", df.duplicated(CLES).sum())
print("Années :", sorted(df["annee"].unique()))
df.columns.tolist()

(349170, 35)
Doublons sur les clés : 0
Années : [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


['code_commune',
 'annee',
 'nb_violences_physiques_intrafamiliales',
 'nb_violences_physiques_hors_cadre_familial',
 'nb_violences_sexuelles',
 'nb_vols_avec_armes',
 'nb_vols_violents_sans_arme',
 'nb_vols_sans_violence_contre_des_personnes',
 'nb_cambriolages_de_logement',
 'nb_vols_de_vehicule',
 'nb_vols_dans_les_vehicules',
 'nb_vols_d_accessoires_sur_vehicules',
 'nb_destructions_et_degradations_volontaires',
 'nb_usage_de_stupefiants',
 'nb_usage_de_stupefiants_afd',
 'nb_trafic_de_stupefiants',
 'nb_escroqueries_et_fraudes_aux_moyens_de_paiement',
 'insee_pop',
 'insee_log',
 'n_indicateurs_estimes',
 'taux_violences_physiques_intrafamiliales',
 'taux_violences_physiques_hors_cadre_familial',
 'taux_violences_sexuelles',
 'taux_vols_avec_armes',
 'taux_vols_violents_sans_arme',
 'taux_vols_sans_violence_contre_des_personnes',
 'taux_cambriolages_de_logement',
 'taux_vols_de_vehicule',
 'taux_vols_dans_les_vehicules',
 'taux_vols_d_accessoires_sur_vehicules',
 'taux_destruction

## 1. Constats de l'EDA utilisés

L'EDA a été faite au bassin de vie. Les constats sont repris ici pour un travail à la commune, où les taux sont moins stables (moins de faits par commune). La cible retenue est le niveau du prix : ce sont les liens avec le niveau (C1) qui guident la sélection.

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Vols de véhicule, vols dans les véhicules | C1 : vols dans les véhicules 0,50 (maisons), le lien le plus fort. B2 : famille la plus territorialisée (η² = 0,49) | Regroupés (même unité : véhicule) |
| Vols d'accessoires sur véhicules | C1 : pas de lien avec le prix (0,08), dilue sa famille | Exclus |
| Vols avec armes, vols violents sans arme | C1 : la famille (0,44) dépasse ses deux indicateurs. B2 : faits très rares (médiane 0,17 pour 1 000) | Regroupés (même unité : infraction) |
| Vols sans violence | C1 : 0,34. B4 : l'indicateur le plus stable (rho = 0,88) | Gardés |
| Cambriolages | B1 : la famille la plus à part. B2 : très territorialisés (η² = 0,42). C1 : lien faible | Gardés (pour 1 000 logements) |
| Destructions et dégradations | C1 : lien faible. B1 : rho = 0,77 avec les vols sans violence | Gardées (redondance à recontrôler à la commune) |
| Usage et trafic de stupéfiants | C1 : lien faible (0,21). B3 : reflètent en partie l'activité des services | Regroupés (même unité : mis en cause) |
| Usage de stupéfiants (AFD) | Déjà compris dans l'usage ; nul avant 2021 | Exclu |
| Violences physiques intrafamiliales | C1 : −0,25 et −0,24. Lien probablement dû aux caractéristiques sociales du territoire | Gardées seules, apport à vérifier en modélisation |
| Violences sexuelles | C1 : −0,36 et −0,30, le lien négatif le plus net | Gardées seules : faits de nature différente des violences physiques, contexte (famille ou extérieur) inconnu |
| Violences hors cadre familial | C1 : pas de lien avec le prix (0,02 et −0,10) | Exclues |
| Escroqueries | B2 : la famille la plus homogène (D9 / D1 = 1,7). B4 : 36 % des écarts entre bassins. C1 : lien faible | Exclues |
| Tous les taux | B3 et B4 : caractéristique stable du territoire ; évolutions nationales dues à des facteurs extérieurs | Pas de variable d'évolution |
| Tous les taux | Distributions asymétriques, valeurs extrêmes (audit, section 7) | À décider : transformation |
| Secret statistique | Faits masqués remplacés par la moyenne départementale (bloc 8) | Indicateur de fiabilité : part des 15 indicateurs estimés |
| Petits territoires | Taux instables quand il y a peu de faits (B3.4 : vols violents, rho = 0,59) | Taux lissé sur 3 ans, calculé après la fusion à la maille choisie (bassin ou sous-bassin) |

## 2. Sélection et regroupement des variables

### 2.1 Regroupement en familles
On additionne les nombres de faits des indicateurs de même nature et de même unité de compte. Aucun taux n'est calculé ici : les taux seront calculés après la fusion avec la table de la maille de modélisation, à partir de la population et de la superficie du territoire. La transformation des variables asymétriques et la mise à l'échelle se feront à ce moment-là.

In [4]:
# Indicateurs gardés, regroupés par famille (même nature, même unité de compte)
FAMILLES = {
    "vols_vehicules":            ["nb_vols_de_vehicule", "nb_vols_dans_les_vehicules"],
    "vols_violents":             ["nb_vols_avec_armes", "nb_vols_violents_sans_arme"],
    "vols_sans_violence":        ["nb_vols_sans_violence_contre_des_personnes"],
    "cambriolages":              ["nb_cambriolages_de_logement"],
    "destructions":              ["nb_destructions_et_degradations_volontaires"],
    "stupefiants":               ["nb_usage_de_stupefiants", "nb_trafic_de_stupefiants"],
    "violences_intrafamiliales": ["nb_violences_physiques_intrafamiliales"],
    "violences_sexuelles":       ["nb_violences_sexuelles"],
}

# Dénominateurs du SSMSI gardés pour calculer les taux après la fusion
fe = df[CLES + ["insee_pop", "insee_log"]].copy()
for famille, colonnes in FAMILLES.items():
    # min_count=1 : la somme reste vide si tous les indicateurs de la famille sont vides
    fe["nb_" + famille] = df[colonnes].sum(axis=1, min_count=1)

print(fe.shape)
print("Valeurs vides par famille :")
print(fe.filter(like="nb_").isna().sum())

(349170, 12)
Valeurs vides par famille :
nb_vols_vehicules               0
nb_vols_violents                0
nb_vols_sans_violence           0
nb_cambriolages                 0
nb_destructions                 0
nb_stupefiants                  0
nb_violences_intrafamiliales    0
nb_violences_sexuelles          0
dtype: int64


### 2.2 Décalage d'un an
Les faits de l'année N servent à expliquer les prix de l'année N+1 : les chiffres d'une année ne sont arrêtés qu'en avril de l'année suivante. La colonne `annee` devient l'année des ventes associées, comme pour LOVAC ; l'année des faits est gardée dans `annee_faits`.

In [5]:
# annee = année des ventes que ces faits servent à expliquer (faits de N -> prix de N+1)
# L'année est lue dans df, qui n'est jamais modifié : la cellule peut être relancée sans risque
fe["annee_faits"] = df["annee"]
fe["annee"] = df["annee"] + 1

print(fe[["annee_faits", "annee"]].drop_duplicates().sort_values("annee").to_string(index=False))

 annee_faits  annee
        2016   2017
        2017   2018
        2018   2019
        2019   2020
        2020   2021
        2021   2022
        2022   2023
        2023   2024
        2024   2025
        2025   2026


### 2.3 Fiabilité
Part des 15 indicateurs estimés à cause du secret statistique : elle sert à repérer les territoires dont les taux sont peu fiables.

In [6]:
# Fiabilité : part des 15 indicateurs estimés à cause du secret statistique
fe["part_indicateurs_estimes"] = df["n_indicateurs_estimes"] / 15 * 100
print("Part des indicateurs estimés (commune) : moyenne =", round(fe["part_indicateurs_estimes"].mean(), 1), "%")

Part des indicateurs estimés (commune) : moyenne = 46.3 %


**Résultat :** en moyenne, 46 % des indicateurs d'une commune sont estimés.

## 3. Variables qualitatives
Aucune variable qualitative. Les valeurs masquées par le secret statistique ont été remplacées par le complément départemental dans la préparation (bloc 8) ; elles sont signalées par la part des indicateurs estimés (section 2.3). Les nombres de faits des communes masquées sont donc des estimations, pas toujours entières.

## 4. Tests statistiques
Reportés après la fusion : ils doivent porter sur les taux. Les nombres de faits suivent la taille de la commune et seraient tous fortement liés entre eux.

## 5. Variables retenues

In [7]:
CLES = ["code_commune", "annee"]   # annee = année des ventes associées
VARIABLES = ["annee_faits", "insee_pop", "insee_log","part_indicateurs_estimes"] + ["nb_" + f for f in FAMILLES]

df_fe = fe[CLES + VARIABLES]

# Contrôle des clés avant l'export
print(df_fe.shape)
print("Doublons sur les clés :", df_fe.duplicated(CLES).sum())

(349170, 14)
Doublons sur les clés : 0


## 6. Fusion des variables retenues avec la cible, export

Rattachement de chaque commune à son sous-bassin (`liaison_bv_scindes`, clé `CODGEO_INI`), agrégation au sous-bassin, puis jointure **à gauche** sur la cible : la cible garde toutes ses lignes (contrôlé par `joindre_cible`).

In [8]:
# Rattachement aux sous-bassins, taux annuels et lissés sur 3 ans
com_delinq = pd.merge(liaison, fe, left_on="CODGEO_INI", right_on="code_commune")
sbv_delinq = taux_lisses(com_delinq, maille="sous_bassin_de_vie", familles=list(FAMILLES))
sbv_delinq["annee_faits"] = sbv_delinq["annee"] - 1

# 1.5 Jointure à la cible et export
joindre_cible(sbv_delinq, ["sous_bassin_de_vie", "annee"], "fe_cible_delinquance_sbv.parquet")

-> Exporté : fe_cible_delinquance_sbv.parquet (21802, 34)


,sous_bassin_de_vie,annee,n_appart,n_maison,prix_m2_med_appart,prix_m2_med_maison,nb_vols_vehicules,nb_vols_violents,nb_vols_sans_violence,nb_cambriolages,...,taux3_cambriolages,taux1_destructions,taux3_destructions,taux1_stupefiants,taux3_stupefiants,taux1_violences_intrafamiliales,taux3_violences_intrafamiliales,taux1_violences_sexuelles,taux3_violences_sexuelles,annee_faits
0,01004,2014,59,368,1836.0,1691.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01004,2015,84,379,1670.0,1667.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01004,2016,76,387,1572.0,1687.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01004,2017,107,479,1727.0,1726.0,236.625140,8.790864,201.739727,214.716364,...,NaN,6.474977,NaN,2.821009,NaN,1.527571,NaN,0.693132,NaN,2016.0
4,01004,2018,118,497,1668.0,1789.0,240.417249,6.662630,227.236301,227.407273,...,NaN,6.019044,NaN,1.916776,NaN,1.771930,NaN,0.743530,NaN,2017.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21797,95652,2021,36,268,3327.0,3342.0,118.699788,15.651750,58.266666,42.909091,...,5.939605,7.143978,7.332218,2.892116,3.176403,1.735241,1.290037,0.875698,0.970230,2020.0
21798,95652,2022,52,211,3301.0,3383.0,78.104782,18.942101,81.561798,44.544554,...,5.701935,7.278473,7.319113,2.639074,2.936721,2.024585,1.675790,0.979860,1.028569,2021.0
21799,95652,2023,44,178,3296.0,3311.0,108.596870,8.591681,85.651163,35.476191,...,4.627511,5.827506,6.746108,3.120272,2.884551,1.980732,1.914163,0.824916,0.893317,2022.0
21800,95652,2024,31,159,3323.0,3182.0,110.823864,11.047864,68.620690,46.020834,...,4.732712,5.496376,6.195444,3.835530,3.201817,2.980126,2.331242,1.001220,0.935375,2023.0


## 7. Après la fusion
- Fait à la section 6 : faits et dénominateurs (`insee_pop`, `insee_log`) additionnés au sous-bassin ; taux annuels (`taux1_`) et lissés sur 3 ans (`taux3_`) pour 1 000 habitants, cambriolages pour 1 000 logements ; fiabilité pondérée par la population. Le taux lissé est vide pour les 2 premières années de faits.
- Reste à faire : tests entre les taux (redondance vols sans violence / destructions), choix de la transformation des taux asymétriques ; choix entre `taux1_` et `taux3_`.
- Mise à l'échelle juste avant la modélisation, avec des paramètres calculés sur les années d'apprentissage seulement.

# Feature engineering – LOVAC (logements privés vacants)
**Contributeurs :** Cécile & Théophile · **Date :** 26/10/2026, mis à jour le 29/10/2026

Préparation à la commune (photo fixe d'un millésime) ; agrégation au sous-bassin à la section 6. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA).

## 0. Chargement
Table produite par le bloc 9 de la préparation : une ligne par commune et par millésime (2020 à 2026). Le millésime N décrit la situation au 1er janvier N-1.

In [9]:
df = pd.read_parquet(DATA_OUT / "lovac_commune_annee.parquet")

# Contrôle : une seule ligne par commune et par millésime
print(df.shape)
print("Doublons :", df.duplicated(["code_commune", "millesime"]).sum())
print("Valeurs vides :")
print(df[["vacants", "vacants_2ans", "parc_prive"]].isna().groupby(df["millesime"]).sum())

(244384, 18)
Doublons : 0
Valeurs vides :
           vacants  vacants_2ans  parc_prive
millesime                                   
2020          7396         20278       18599
2021          7184         20030       18521
2022          7459         20343       18515
2023          7202         19826       18485
2024          6866         26927       18401
2025          9395         22135       18321
2026          6882         17767       34912


## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| `taux_vacance_complete` | C4 : ρ = −0,44 (maisons), −0,33 (appartements). Parc privé caché pour environ 53 % des communes, surtout les petites (B2.7 : 85 % des communes de moins de 100 logements ; 90,6 % des logements sont dans une commune au parc connu) | Gardé ; taux caché complété par celui de l'EPCI (préparation, bloc 9) |
| `taux_impute` | Signale les taux complétés par l'EPCI | Gardé pour le contrôle qualité après la jointure |
| `taux_vacance_2ans` | C4 : ρ = −0,70 et −0,57, mais environ 85 % de valeurs manquantes, non complétées : lien mesuré sur les seules communes visibles, surtout les plus grandes | Écartée a priori. Variable à suivre pour les grandes agglomérations, quand elles seront séparées de leur bassin de vie |
| `part_longue_parmi_vacants` | C4 : doublon de `taux_vacance_2ans` (0,88) | Exclue |
| Millésime 2026 | Pas de parc privé : taux impossible | Hors apprentissage et test ; sert seulement à la simulation finale |
| Tous les millésimes | Classement des communes stable entre millésimes (B2.6 : ρ de 0,87 à 0,92) ; la cible est le niveau du prix | Une seule photo pour toutes les années, comme le recensement (plan 5.3) |
| Rupture de série | B2.6 : la chute nationale apparaît dans le millésime 2025 (situation 2024 : 7,2 % contre environ 11 %) | Photo du millésime 2023, avant la rupture ; elle couvre aussi les prix 2014-2019, sans LOVAC sinon |

## 2. Sélection des variables
On garde le taux de vacance complété par l'EPCI et l'indicateur de complétion, pour tous les millésimes ; la photo du millésime 2023 est extraite à la section 5.

In [10]:
CLES = ["code_commune", "annee"]   # annee = année des ventes associées

fe = df.rename(columns={"annee_vente_associee": "annee"})
fe = fe[CLES + ["millesime", "taux_vacance_complete", "taux_impute", "apres_rupture"]].copy()

# Contrôle : clés uniques, valeurs vides par année
print(fe.shape)
print("Doublons sur les clés :", fe.duplicated(CLES).sum())
print(fe.groupby("annee")[["taux_vacance_complete"]].apply(lambda g: g.isna().sum()))

(244384, 6)
Doublons sur les clés : 0
       taux_vacance_complete
annee                       
2020                       0
2021                       0
2022                       0
2023                       0
2024                       0
2025                       0
2026                   34912


## 3. Variables qualitatives
Aucune à encoder : `taux_impute` est déjà binaire (vrai / faux). `apres_rupture` n'est plus utile avec une photo fixe d'avant la rupture.

## 4. Tests statistiques
Reportés après la fusion : ils doivent porter sur les taux à la maille finale (redondance avec la vacance du recensement, part des logements dont le taux est complété par l'EPCI).

## 5. Variables retenues
Photo fixe : millésime 2023 (situation au 1er janvier 2022), recopiée pour toutes les années de vente à la fusion. La clé est donc `code_commune`, sans année.

In [11]:
MILLESIME_PHOTO_LOVAC = 2023      # avant la rupture de série (millésime 2025)

df_fe = fe.loc[fe["millesime"] == MILLESIME_PHOTO_LOVAC,
               ["code_commune", "taux_vacance_complete", "taux_impute"]]

# Contrôles : une ligne par commune, pas de taux vide
print(df_fe.shape)
print("Doublons :", df_fe["code_commune"].duplicated().sum(),
      "| taux vides :", df_fe["taux_vacance_complete"].isna().sum())

(34912, 3)
Doublons : 0 | taux vides : 0


## 6. Fusion des variables retenues avec la cible, export

Rattachement de chaque commune à son sous-bassin (`liaison_bv_scindes`, clé `CODGEO_INI`), agrégation au sous-bassin, puis jointure **à gauche** sur la cible : la cible garde toutes ses lignes (contrôlé par `joindre_cible`).

In [12]:
p22_log = pd.read_parquet(DATA_OUT / "logement_commune_2022.parquet", columns=["code_commune", "P22_LOG"])

# 2.2 Poids : nombre de logements du recensement (parc privé inconnu pour les communes masquées)
fe_lovac = pd.merge(df_fe, p22_log, on="code_commune", how="left")
com_lovac = pd.merge(liaison, fe_lovac, left_on="CODGEO_INI", right_on="code_commune")

# 2.3 Moyenne pondérée par P22_LOG au sous-bassin
com_lovac["vacance_x_log"] = com_lovac["taux_vacance_complete"] * com_lovac["P22_LOG"]
com_lovac["impute_x_log"] = com_lovac["taux_impute"].astype(float) * com_lovac["P22_LOG"]
sbv_lovac = com_lovac.groupby("sous_bassin_de_vie")[["vacance_x_log", "impute_x_log", "P22_LOG"]].sum()
sbv_lovac["taux_vacance_complete"] = sbv_lovac["vacance_x_log"] / sbv_lovac["P22_LOG"]
sbv_lovac["part_logements_taux_impute"] = sbv_lovac["impute_x_log"] / sbv_lovac["P22_LOG"] * 100
sbv_lovac["millesime_lovac"] = MILLESIME_PHOTO_LOVAC
sbv_lovac = sbv_lovac[["taux_vacance_complete", "part_logements_taux_impute", "millesime_lovac"]].reset_index()

# 2.4 Jointure à la cible (même photo pour toutes les années) et export
joindre_cible(sbv_lovac, ["sous_bassin_de_vie"], "fe_cible_vacance_sbv.parquet")

-> Exporté : fe_cible_vacance_sbv.parquet (21802, 9)


,sous_bassin_de_vie,annee,n_appart,n_maison,prix_m2_med_appart,prix_m2_med_maison,taux_vacance_complete,part_logements_taux_impute,millesime_lovac
0,01004,2014,59,368,1836.0,1691.0,9.955718,6.432817,2023
1,01004,2015,84,379,1670.0,1667.0,9.955718,6.432817,2023
2,01004,2016,76,387,1572.0,1687.0,9.955718,6.432817,2023
3,01004,2017,107,479,1727.0,1726.0,9.955718,6.432817,2023
4,01004,2018,118,497,1668.0,1789.0,9.955718,6.432817,2023
...,...,...,...,...,...,...,...,...,...
21797,95652,2021,36,268,3327.0,3342.0,9.946787,7.490924,2023
21798,95652,2022,52,211,3301.0,3383.0,9.946787,7.490924,2023
21799,95652,2023,44,178,3296.0,3311.0,9.946787,7.490924,2023
21800,95652,2024,31,159,3323.0,3182.0,9.946787,7.490924,2023


## 7. Après la fusion
- Fait à la section 6 : moyenne des taux communaux pondérée par le nombre de logements du recensement (`P22_LOG`) ; le parc privé est inconnu pour les communes masquées et les logements du recensement comprennent le parc social (approximation). `part_logements_taux_impute` = part des logements du sous-bassin situés dans une commune dont le taux a été complété par l'EPCI (contrôle qualité).
- Même photo (millésime 2023) pour toutes les années de vente.
- Reste à faire : redondance avec le taux de vacance du recensement ; transformation éventuelle et mise à l'échelle juste avant la modélisation (paramètres calculés sur les années d'apprentissage seulement).

# Feature engineering – Équipements (BPE 2025 par commune)
**Contributeurs :** Cécile & Théophile · **Date :** 27/10/2026, mis à jour le 29/10/2026

Préparation à la commune (table du bloc 12) ; somme au sous-bassin à la section 6. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA complémentaire, faite sur la BPE en évolution au bassin de vie).

## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Source | Audit : BPE en évolution, nombres d'équipements comparables en 2015, 2020 et 2025, directement au bassin de vie | EDA faite sur cette source ; pour le découpage en sous-bassins, remplacée par la BPE 2025 par commune (bloc 12), types communs |
| Enseignement | Audit : 3 types, équipements de l'Éducation nationale absents | Exclu |
| Six autres domaines | B1 : rho entre domaines ≤ 0,64, pas de redondance. C1 ter : tous liés au prix à densité égale (rho partiels de 0,22 à 0,57 ; sports −0,26) | Gardés, sports compris |
| Millésime | B3 et B4 : rho de 0,82 à 0,99 entre millésimes ; 83 à 99 % des écarts viennent des différences entre bassins. Interpolation avec 2025 = information publiée après la date de prédiction | Photo fixe (BPE 2025, seul millésime disponible à la commune) pour toutes les années, pas de variable d'évolution |
| Forme | C1 : le taux par habitant mesure aussi le caractère rural. C1 bis : les équipements par km² suivent la densité (rho de 0,81 à 0,96) | Taux pour 10 000 habitants + densité de population, calculés après la fusion |

## 2. Sélection et création des variables

### 2.1 Nombre d'équipements par commune (BPE2026) 
Table produite par le bloc 12 de la préparation : nombre d'équipements par commune et par domaine, types communs avec la BPE en évolution. Elle remplace la BPE en évolution au bassin de vie, qui ne peut pas être découpée en sous-bassins.

In [13]:
equip = pd.read_parquet(DATA_OUT / "equipements_commune_2025.parquet")

print(equip.shape, "| doublons :", equip["code_commune"].duplicated().sum())
equip.describe().round(0)

(34786, 8) | doublons : 0


,nb_equip_commerces,nb_equip_sante,nb_equip_services,nb_equip_sports,nb_equip_tourisme,nb_equip_transports,millesime_bpe
count,34786.0,34786.0,34786.0,34786.0,34786.0,34786.0,34786.0
mean,10.0,11.0,37.0,5.0,1.0,3.0,2025.0
std,63.0,74.0,180.0,14.0,8.0,26.0,0.0
min,0.0,0.0,0.0,0.0,0.0,0.0,2025.0
25%,0.0,0.0,4.0,1.0,0.0,0.0,2025.0
50%,1.0,0.0,8.0,2.0,0.0,0.0,2025.0
75%,4.0,3.0,22.0,6.0,1.0,1.0,2025.0
max,2898.0,4158.0,12078.0,843.0,493.0,1652.0,2025.0


### 2.2 Contrôle : chaque commune se rattache-t-elle à un sous-bassin ?

In [14]:
controle = equip.merge(liaison[["CODGEO_INI", "BV2022", "sous_bassin_de_vie"]], left_on="code_commune",
                       right_on="CODGEO_INI", how="left", validate="one_to_one")

print("Communes sans sous-bassin :", controle["sous_bassin_de_vie"].isna().sum())
print("Sous-bassins couverts :", controle["sous_bassin_de_vie"].nunique(),
      "| bassins de vie couverts :", controle["BV2022"].nunique())

Communes sans sous-bassin : 0
Sous-bassins couverts : 1894 | bassins de vie couverts : 1681


## 3. Variables qualitatives
Aucune.

## 4. Tests statistiques
La table du bloc 12 compte les équipements de chaque commune : elle n'a aucune case vide (0 quand un domaine est absent). Une commune absente de la table n'a aucun équipement recensé ; son nombre vaut 0 à la fusion. Cette règle s'appuie sur le test de l'EDA complémentaire (partie A) : dans la BPE en évolution, les cases vides concernaient de petits bassins (4 215 et 3 801 logements médians contre 9 093, Welch p < 0,001, d = −1,41 et −1,36) et signifiaient « aucun équipement de ce domaine ».


## 5. Variables retenues

In [15]:
print("Équipements :", equip.shape, "| doublons :", equip["code_commune"].duplicated().sum())

print(equip.columns)

Équipements : (34786, 8) | doublons : 0
Index(['code_commune', 'nb_equip_commerces', 'nb_equip_sante',
       'nb_equip_services', 'nb_equip_sports', 'nb_equip_tourisme',
       'nb_equip_transports', 'millesime_bpe'],
      dtype='str')


## 6. Fusion des variables retenues avec la cible, export

Rattachement de chaque commune à son sous-bassin (`liaison_bv_scindes`, clé `CODGEO_INI`), agrégation au sous-bassin, puis jointure **à gauche** sur la cible : la cible garde toutes ses lignes (contrôlé par `joindre_cible`).

In [16]:
cols_equip = [c for c in equip.columns if c.startswith("nb_equip_")]

# 3.1 Rattachement et somme par sous-bassin
com_equip = pd.merge(liaison, equip, left_on="CODGEO_INI", right_on="code_commune")
sbv_equip = com_equip.groupby("sous_bassin_de_vie")[cols_equip].sum().reset_index()

# 3.2 Jointure à la cible ; sous-bassin sans équipement recensé = 0 (règle de l'EDA complémentaire)
final_equip = pd.merge(cible, sbv_equip, on="sous_bassin_de_vie", how="left", validate="many_to_one")
final_equip[cols_equip] = final_equip[cols_equip].fillna(0)
final_equip["millesime_bpe"] = 2025
assert len(final_equip) == len(cible), "Nombre de lignes modifié par la jointure : équipements"
final_equip.to_parquet(DATA_OUT / "fe_cible_equipement_sbv.parquet", index=False)
print("-> Exporté : fe_cible_equipement_sbv.parquet", final_equip.shape)

-> Exporté : fe_cible_equipement_sbv.parquet (21802, 13)


## 7. Après la fusion
- Fait à la section 6 : nombres d'équipements additionnés au sous-bassin ; sous-bassin sans équipement recensé = 0 ; `millesime_bpe` = 2025 pour toutes les années de vente.
- Reste à faire : taux pour 10 000 habitants (population additionnée par sous-bassin) et densité de population (superficie récupérée hors de ce fichier).
- Limite : la BPE 2025 décrit une année postérieure aux ventes d'apprentissage ; acceptable car le classement des territoires est stable entre millésimes (rho de 0,82 à 0,99, EDA complémentaire).
- Transformation éventuelle et mise à l'échelle juste avant la modélisation (paramètres calculés sur les années d'apprentissage seulement).

# Feature engineering – Parc de logements (recensement)
**Contributeurs :** Cécile & Théophile · **Date :** 27/10/2026, mis à jour le 29/10/2026

Préparation à la commune ; effectifs additionnés au sous-bassin et indicateurs calculés à la section 6. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA et l'EDA complémentaire).

## 0. Chargement
Table produite par le bloc 7 de la préparation : une ligne par commune (Paris, Lyon, Marseille par arrondissement), recensement 2022, effectifs bruts et indicateurs.

In [17]:
df = pd.read_parquet(DATA_OUT / "logement_commune_2022.parquet")

print(df.shape)
print("Doublons :", df.duplicated("code_commune").sum())

(34955, 52)
Doublons : 0


## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Sélection | Rapport, section 4.8 : 12 indicateurs gardés (\|rho\| > 0,2 à la commune, doublons > 0,8 écartés) | Les 12 gardés |
| Millésime | EDA complémentaire : de 2016 à 2022, le choix du millésime fait varier le rho avec le prix de 0,06 au plus ; les bassins gardent leur classement | Photo 2022 pour toutes les années, pas de variable d'évolution |
| Taux de vacance | EDA complémentaire : rho de −0,76 au bassin de vie (maisons et appartements) | Gardé, à côté du taux de LOVAC (redondance à contrôler après la fusion) |
| Taille des ménages | Même définition que la taille moyenne des ménages du bloc 6 | Gardée ; une seule des deux à retenir après la fusion |
| Effectifs | Au bassin de vie, les parts se recalculent en additionnant les effectifs | Effectifs bruts exportés, indicateurs calculés après la fusion |

## 2. Sélection des variables
Pour chacun des 12 indicateurs, on garde les effectifs du numérateur et du dénominateur.

In [18]:
# Effectifs nécessaires à chaque indicateur retenu (numérateur puis dénominateur)
INDICATEURS_LOGEMENT = {
    "taux_vacance":       ["P22_LOGVAC", "P22_LOG"],
    "part_rsec":          ["P22_RSECOCC", "P22_LOG"],
    "part_apparts":       ["P22_APPART", "P22_LOG"],
    "log_nb_logements":   ["P22_LOG"],
    "part_moins40m2":     ["P22_RP_M30M2", "P22_RP_3040M2", "P22_RP"],
    "part_avant1946":     ["P22_RP_ACH1919", "P22_RP_ACH1945", "P22_RP"],
    "part_2006_2019":     ["P22_RP_ACH2019", "P22_RP"],
    "anciennete_moyenne": ["P22_ANEM_RP", "P22_RP"],
    "taille_menage":      ["P22_NPER_RP", "P22_RP"],
    "part_sans_voiture":  ["P22_RP_VOIT1P", "P22_MEN"],
    "part_parking":       ["P22_RP_GARL", "P22_MEN"],
    "part_suroccupes":    ["C22_RP_SUROCC_MOD", "C22_RP_SUROCC_ACC", "P22_RP"],
}
EFFECTIFS_LOGEMENT = sorted({c for cols in INDICATEURS_LOGEMENT.values() for c in cols})

fe = df[["code_commune", "annee_rp"] + EFFECTIFS_LOGEMENT].copy()
print(fe.shape)
print("Valeurs vides :", fe[EFFECTIFS_LOGEMENT].isna().sum().sum())

(34955, 19)
Valeurs vides : 0


## 3. Variables qualitatives
Aucune. Les colonnes d'aide de la préparation (Alsace-Moselle, petites communes) ne sont pas reprises : l'Alsace-Moselle sort à la jointure avec la cible, et le seuil de 20 ventes s'applique à la maille de modélisation.

## 4. Tests statistiques
Faits dans le rapport (section 4.8, à la commune) et dans l'EDA complémentaire (au bassin de vie) ; à refaire sur les indicateurs après la fusion.

## 5. Variables retenues et export

In [19]:
print(fe.shape, "| doublons :", fe.duplicated("code_commune").sum())

(34955, 19) | doublons : 0


## 6. Fusion des variables retenues avec la cible, export

Rattachement de chaque commune à son sous-bassin (`liaison_bv_scindes`, clé `CODGEO_INI`), agrégation au sous-bassin, puis jointure **à gauche** sur la cible : la cible garde toutes ses lignes (contrôlé par `joindre_cible`).

In [20]:
# 4.2 Effectifs additionnés au sous-bassin, puis les 12 indicateurs
com_logement = pd.merge(liaison, fe[["code_commune"] + EFFECTIFS_LOGEMENT],
                        left_on="CODGEO_INI", right_on="code_commune")
sbv_logement = indicateurs_logement(com_logement.groupby("sous_bassin_de_vie")[EFFECTIFS_LOGEMENT].sum())
sbv_logement["millesime_rp"] = 2022
sbv_logement = sbv_logement.reset_index()

# 4.3 Jointure à la cible (même photo pour toutes les années) et export
joindre_cible(sbv_logement, ["sous_bassin_de_vie"], "fe_cible_logement_sbv.parquet")

-> Exporté : fe_cible_logement_sbv.parquet (21802, 19)


,sous_bassin_de_vie,annee,n_appart,n_maison,prix_m2_med_appart,prix_m2_med_maison,taux_vacance,part_rsec,part_apparts,log_nb_logements,part_moins40m2,part_avant1946,part_2006_2019,anciennete_moyenne,taille_menage,part_sans_voiture,part_parking,part_suroccupes,millesime_rp
0,01004,2014,59,368,1836.0,1691.0,8.619038,5.523252,28.264808,4.396341,4.446464,22.882237,19.924555,14.192627,2.235500,9.289928,77.151416,3.816036,2022
1,01004,2015,84,379,1670.0,1667.0,8.619038,5.523252,28.264808,4.396341,4.446464,22.882237,19.924555,14.192627,2.235500,9.289928,77.151416,3.816036,2022
2,01004,2016,76,387,1572.0,1687.0,8.619038,5.523252,28.264808,4.396341,4.446464,22.882237,19.924555,14.192627,2.235500,9.289928,77.151416,3.816036,2022
3,01004,2017,107,479,1727.0,1726.0,8.619038,5.523252,28.264808,4.396341,4.446464,22.882237,19.924555,14.192627,2.235500,9.289928,77.151416,3.816036,2022
4,01004,2018,118,497,1668.0,1789.0,8.619038,5.523252,28.264808,4.396341,4.446464,22.882237,19.924555,14.192627,2.235500,9.289928,77.151416,3.816036,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21797,95652,2021,36,268,3327.0,3342.0,5.633270,1.951293,22.560867,3.948980,9.616039,20.885788,15.432192,15.441779,2.453549,7.609169,77.472047,7.058735,2022
21798,95652,2022,52,211,3301.0,3383.0,5.633270,1.951293,22.560867,3.948980,9.616039,20.885788,15.432192,15.441779,2.453549,7.609169,77.472047,7.058735,2022
21799,95652,2023,44,178,3296.0,3311.0,5.633270,1.951293,22.560867,3.948980,9.616039,20.885788,15.432192,15.441779,2.453549,7.609169,77.472047,7.058735,2022
21800,95652,2024,31,159,3323.0,3182.0,5.633270,1.951293,22.560867,3.948980,9.616039,20.885788,15.432192,15.441779,2.453549,7.609169,77.472047,7.058735,2022


## 7. Après la fusion
- Fait à la section 6 : effectifs additionnés au sous-bassin, puis les 12 indicateurs (`indicateurs_logement`, formules du bloc 7 ; `log_nb_logements` = log10 du nombre de logements). Même photo (2022) pour toutes les années de vente.
- Reste à faire : redondances entre indicateurs, en priorité part des moins de 40 m² / part des logements suroccupés, part sans voiture / part avec parking ; taux de vacance du recensement / de LOVAC ; taille des ménages (logement) / taille moyenne des ménages (ménages).
- Transformation éventuelle et mise à l'échelle juste avant la modélisation (paramètres calculés sur les années d'apprentissage seulement).

# Feature engineering – Filosofi (niveau de vie)
**Contributeurs :** Cécile & Théophile · **Date :** 27/10/2026, mis à jour le 29/10/2026

Préparation à la commune ; agrégation au sous-bassin à la section 6. La cible n'est pas utilisée ici (déjà étudiée dans l'EDA Filosofi).

## 0. Chargement
Table produite par le bloc 5 de la préparation : une ligne par commune et par année, de 2014 à 2021, chaque millésime dans sa propre géographie.

In [21]:
df = pd.read_parquet(DATA_OUT / "filosofi_commune_2014_2021.parquet")

print(df.shape)
print("Doublons commune x année :", df.duplicated(["code_commune", "annee"]).sum())
print("Années :", sorted(df["annee"].unique()))

(261319, 29)
Doublons commune x année : 0
Années : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]


## 1. Constats de l'EDA utilisés

| Variable | Constat de l'EDA | Traitement |
|---|---|---|
| Niveau de vie médian | C1 : lien fort avec le niveau des prix, rho = 0,66 (maisons et appartements), stable de 0,61 à 0,66 sur 2014-2021 | Gardé (la cible retenue le 26/10 est le niveau du prix) |
| Niveau de vie médian | B3 : très stable, rho = 0,90 entre 2014 et 2021 ; 88 % des écarts viennent des différences entre communes | Une photo suffit : millésime 2021 pour toutes les années de vente |
| Indicateurs détaillés (Gini, déciles, parts de revenus) | Masqués pour environ 84 % des communes ; Gini fragile ; non recalculables au territoire | Exclus |
| Valeurs masquées | En 2021, médiane masquée pour 3 604 communes sur 34 926 (10 %), surtout de très petites communes ; leur nombre de ménages fiscaux est masqué aussi | Laissées vides ; communes non comptées dans le calcul au territoire ; part des ménages concernés à contrôler avec le recensement (`P22_MEN`) |
| Paris, Lyon, Marseille | Présentes deux fois : ville entière et arrondissements | Ville entière retirée, arrondissements gardés |

## 2. Sélection des variables
Niveau de vie médian de 2021 et nombre de ménages fiscaux, qui servira de poids pour passer au territoire (on ne peut pas additionner des médianes).

In [22]:
MILLESIME_FILOSOFI = 2021   # photo : millésime le plus récent

fe = df.loc[df["annee"] == MILLESIME_FILOSOFI, ["code_commune", "niveau_vie_median", "NBMEN"]].copy()
# Paris, Lyon, Marseille : arrondissements gardés, ville entière retirée (sinon comptée deux fois)
fe = fe[~fe["code_commune"].isin(["75056", "69123", "13055"])]
fe = fe.rename(columns={"NBMEN": "nb_menages_fiscaux"})

print(fe.shape)
print("Doublons :", fe.duplicated("code_commune").sum(),
      "| médianes masquées :", fe["niveau_vie_median"].isna().sum())
fe.describe().round(0)

(34926, 3)
Doublons : 0 | médianes masquées : 3604


,niveau_vie_median,nb_menages_fiscaux
count,31322.0,31322.0
mean,23244.0,915.0
std,3174.0,4044.0
min,12260.0,35.0
25%,21270.0,109.0
50%,22810.0,226.0
75%,24710.0,556.0
max,55340.0,222023.0


## 3. Variables qualitatives
Aucune.

## 4. Tests statistiques
Faits dans l'EDA Filosofi (B3, C1) ; à refaire au territoire après la fusion.

## 5. Variables retenues

In [23]:
print(fe.shape)

(34926, 3)


## 6. Fusion des variables retenues avec la cible, export

Rattachement de chaque commune à son sous-bassin (`liaison_bv_scindes`, clé `CODGEO_INI`), agrégation au sous-bassin, puis jointure **à gauche** sur la cible : la cible garde toutes ses lignes (contrôlé par `joindre_cible`).

In [24]:
# 5.2 Moyenne des médianes communales pondérée par les ménages fiscaux,
#     sur les seules communes dont la médiane ET le nombre de ménages sont connus
com_filosofi = pd.merge(liaison, fe, left_on="CODGEO_INI", right_on="code_commune")
connue = com_filosofi["niveau_vie_median"].notna() & com_filosofi["nb_menages_fiscaux"].notna()
com_filosofi["poids"] = com_filosofi["nb_menages_fiscaux"].where(connue)
com_filosofi["masse_niveau_vie"] = (com_filosofi["niveau_vie_median"] * com_filosofi["nb_menages_fiscaux"]).where(connue)
sbv_filosofi = com_filosofi.groupby("sous_bassin_de_vie")[["masse_niveau_vie", "poids"]].sum(min_count=1)
sbv_filosofi["niveau_vie_median"] = sbv_filosofi["masse_niveau_vie"] / sbv_filosofi["poids"]
sbv_filosofi = sbv_filosofi.rename(columns={"poids": "nb_menages_fiscaux"})
sbv_filosofi["millesime_filosofi"] = MILLESIME_FILOSOFI
sbv_filosofi = sbv_filosofi[["niveau_vie_median", "nb_menages_fiscaux", "millesime_filosofi"]].reset_index()

# 5.3 Jointure à la cible (même photo pour toutes les années) et export
joindre_cible(sbv_filosofi, ["sous_bassin_de_vie"], "fe_cible_filosofi_sbv.parquet")

-> Exporté : fe_cible_filosofi_sbv.parquet (21802, 9)


,sous_bassin_de_vie,annee,n_appart,n_maison,prix_m2_med_appart,prix_m2_med_maison,niveau_vie_median,nb_menages_fiscaux,millesime_filosofi
0,01004,2014,59,368,1836.0,1691.0,23169.329383,20757.0,2021
1,01004,2015,84,379,1670.0,1667.0,23169.329383,20757.0,2021
2,01004,2016,76,387,1572.0,1687.0,23169.329383,20757.0,2021
3,01004,2017,107,479,1727.0,1726.0,23169.329383,20757.0,2021
4,01004,2018,118,497,1668.0,1789.0,23169.329383,20757.0,2021
...,...,...,...,...,...,...,...,...,...
21797,95652,2021,36,268,3327.0,3342.0,27745.532845,7779.0,2021
21798,95652,2022,52,211,3301.0,3383.0,27745.532845,7779.0,2021
21799,95652,2023,44,178,3296.0,3311.0,27745.532845,7779.0,2021
21800,95652,2024,31,159,3323.0,3182.0,27745.532845,7779.0,2021


## 7. Après la fusion
- Fait à la section 6 : moyenne des médianes communales pondérée par le nombre de ménages fiscaux, sur les seules communes dont la médiane et le nombre de ménages sont connus (approximation : une moyenne de médianes n'est pas la médiane du territoire). `nb_menages_fiscaux` = ménages des communes prises en compte.
- Reste à faire : part des ménages du sous-bassin dans une commune à médiane masquée (comparer `nb_menages_fiscaux` au `P22_MEN` du recensement additionné).
- Limite à rappeler : le millésime 2021, publié en 2024, est postérieur aux ventes d'apprentissage ; sans effet si le classement des territoires est stable (EDA, B3).

# Feature engineering – DVF+ (volume de ventes)
**Contributeurs :** Cécile & Théophile· **Date :** 27/10/2026, mis à jour le 29/10/2026

Préparation à la commune × année ; somme au sous-bassin × année à la section 6. La cible n'est pas utilisée ici.

## 0. Chargement
Table produite par le bloc 4 de la préparation : nombre de ventes et prix médian au m² par commune (géographie 2026, arrondissements séparés) et par année, maisons et appartements, toutes les ventes retenues (pas l'échantillon).

In [25]:
df = pd.read_parquet(DATA_OUT / "cible_commune_annee.parquet")

print(df.shape)
print("Doublons :", df.duplicated(["code_commune", "annee"]).sum())
print("Années :", sorted(df["annee"].unique()))

(362187, 6)
Doublons : 0
Années : [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 1. Constats utilisés

| Variable | Constat | Traitement |
|---|---|---|
| Nombre de ventes | Volet séries temporelles : le marché s'ajuste d'abord par les volumes puis par les prix (test de Granger : le volume précède le prix des maisons, p = 0,001). Seule variable du projet qui varie à la fois d'un territoire à l'autre et d'une année à l'autre | Gardé, maisons et appartements séparés |
| Nombre de ventes de l'année N | Inconnu au moment de prévoir le prix de N | Décalage d'un an : les ventes de N-1 servent pour le prix de N |
| Prix médian | C'est la cible | Exclu de ce fichier |

## 2. Sélection des variables et décalage
Nombres de ventes seulement : ils s'additionnent à n'importe quelle maille. L'évolution du volume (N-2 → N-1) sera calculée après la fusion, sur les nombres additionnés au territoire.

In [26]:
CLES = ["code_commune", "annee"]   # annee = année des ventes que ces volumes servent à expliquer

fe = df[["code_commune", "annee", "n_maison", "n_appart"]].rename(
    columns={"n_maison": "n_ventes_maison_n1", "n_appart": "n_ventes_appart_n1"})

# annee = année du prix (ventes de N-1 -> prix de N) ; l'année des ventes est gardée
fe["annee_ventes"] = df["annee"]
fe["annee"] = df["annee"] + 1

print(fe.shape)
print("Doublons sur les clés :", fe.duplicated(CLES).sum())
print(fe[["annee_ventes", "annee"]].drop_duplicates().sort_values("annee").to_string(index=False))

(362187, 5)


Doublons sur les clés : 0
 annee_ventes  annee
         2014   2015
         2015   2016
         2016   2017
         2017   2018
         2018   2019
         2019   2020
         2020   2021
         2021   2022
         2022   2023
         2023   2024
         2024   2025
         2025   2026


## 3. Variables qualitatives
Aucune.

## 4. Tests statistiques
Reportés après la fusion : ils doivent porter sur les volumes additionnés au territoire et sur leur évolution.

## 5. Variables retenues

In [27]:
VARIABLES = ["annee_ventes", "n_ventes_maison_n1", "n_ventes_appart_n1"]

df_fe = fe[CLES + VARIABLES]

print(df_fe.shape)
print("Doublons sur les clés :", df_fe.duplicated(CLES).sum())

(362187, 5)
Doublons sur les clés : 0


## 6. Fusion des variables retenues avec la cible, export

Rattachement de chaque commune à son sous-bassin (`liaison_bv_scindes`, clé `CODGEO_INI`), agrégation au sous-bassin, puis jointure **à gauche** sur la cible : la cible garde toutes ses lignes (contrôlé par `joindre_cible`). Les années sans vente valent 0, sauf celles dont l'année N-1 n'existe pas dans DVF+ (prix 2014 : pas de ventes 2013), laissées vides.

In [28]:
# 6.2 Rattachement et somme par sous-bassin x année
com_dvf = pd.merge(liaison, df_fe, left_on="CODGEO_INI", right_on="code_commune")
cols_ventes = ["n_ventes_maison_n1", "n_ventes_appart_n1"]
sbv_dvf = com_dvf.groupby(["sous_bassin_de_vie", "annee"])[cols_ventes].sum().reset_index()

# 6.3 Jointure à la cible ; aucune vente = 0, mais seulement pour les années
#     dont l'année N-1 existe dans DVF+ (pas de ventes 2013 : prix 2014 laissés vides)
final_dvf = pd.merge(cible, sbv_dvf, on=["sous_bassin_de_vie", "annee"], how="left", validate="one_to_one")
annee_n1_connue = final_dvf["annee"] >= df["annee"].min() + 1
final_dvf.loc[annee_n1_connue, cols_ventes] = final_dvf.loc[annee_n1_connue, cols_ventes].fillna(0)
final_dvf["annee_ventes"] = final_dvf["annee"] - 1
assert len(final_dvf) == len(cible), "Nombre de lignes modifié par la jointure : DVF+"
final_dvf.to_parquet(DATA_OUT / "fe_cible_dvf_sbv.parquet", index=False)
print("-> Exporté : fe_cible_dvf_sbv.parquet", final_dvf.shape)

-> Exporté : fe_cible_dvf_sbv.parquet (21802, 9)


## 7. Après la fusion
- Fait à la section 6 : nombres de ventes additionnés au sous-bassin ; année sans vente = 0, sauf 2014 (pas de ventes 2013) laissée vide.
- Reste à faire : évolution du volume entre N-2 et N-1, par type de bien : (ventes N-1 − ventes N-2) / ventes N-2, à partir des nombres additionnés.
- Ventes de 2025 (prix 2026) : hors apprentissage et test, réservées à la simulation finale.

# Feature engineering – Taux des crédits immobiliers
**Contributeurs :** Cécile & Théophile· **Date :** 28/10/2026

Maille : France entière, une valeur par mois (Banque de France). Table produite par le bloc 11 de la préparation (mois, trimestre, année).

## 1. Constats utilisés

| Constat | Source | Conséquence |
|---|---|---|
| Le taux est le même pour tous les territoires un mois donné | Bloc 11 | Dans une table territoire × année, il ne prend qu'une valeur par année |
| De 2020 à 2023, le modèle ne voit que 4 valeurs du taux, toutes basses | Baseline | Il ne peut pas séparer l'effet du taux de tout ce qui s'est passé la même année |
| En 2024 et 2025, le taux sort de la plage vue en apprentissage | Baseline | Une régression prolonge sa droite, un arbre répète la dernière valeur vue : la prévision n'est pas fiable |
| Ajouté à la baseline, le taux décale toutes les prévisions de 2024 d'environ 890 €/m² | Baseline (SHAP) | Il dégrade la prévision au lieu de l'améliorer |
| À la maille mensuelle, le taux apporte peu à la prévision du prix | Volet séries temporelles (SARIMAX) | Même avec une valeur par mois, son apport reste faible |

## 2. Décision
Le taux des crédits n'entre pas dans la modélisation annuelle par territoire. Il est étudié seulement dans le volet séries temporelles (SARIMAX), à la maille mensuelle, où il prend une valeur différente chaque mois.

Aucun fichier exporté.

# Table finale – toutes les sources


Jointure des six tables `fe_cible_<source>_sbv.parquet` sur sous-bassin × année, toujours à gauche sur la cible : la table finale a exactement les lignes de la cible.

- Les colonnes de la cible (`n_appart`, `n_maison`, prix médians) ne sont gardées qu'une fois.
- Une variable présente dans deux sources arrête le code au lieu d'être renommée `_x` / `_y` en silence.
- Les redondances statistiques entre sources (vacance LOVAC / recensement, taille des ménages, etc.) ne sont pas traitées ici : elles se contrôlent sur cette table.

Export : `fe_cible_toutes_sources_sbv.parquet`.

In [29]:
cible = pd.read_parquet(DATA_OUT / "cible_sbv_annee.parquet")

# Tables produites par les sections précédentes (une ligne par sous-bassin x année, comme la cible)
FICHIERS_FE = [
    "fe_cible_delinquance_sbv.parquet",
    "fe_cible_vacance_sbv.parquet",
    "fe_cible_equipement_sbv.parquet",
    "fe_cible_logement_sbv.parquet",
    "fe_cible_filosofi_sbv.parquet",
    "fe_cible_dvf_sbv.parquet",
]
CLES_SBV = ["sous_bassin_de_vie", "annee"]
cols_cible = [c for c in cible.columns if c not in CLES_SBV]   # colonnes de la cible, gardées une seule fois
assert not cible.duplicated(CLES_SBV).any(), "La cible a des doublons sur sous-bassin x année"

table_finale = cible.copy()
for nom in FICHIERS_FE:
    t = pd.read_parquet(DATA_OUT / nom).drop(columns=cols_cible)
    # Une colonne présente dans deux sources serait renommée _x / _y en silence : on l'interdit
    deja_la = (set(t.columns) & set(table_finale.columns)) - set(CLES_SBV)
    assert not deja_la, f"{nom} : colonnes déjà présentes {sorted(deja_la)}"
    table_finale = pd.merge(table_finale, t, on=CLES_SBV, how="left", validate="one_to_one")

assert len(table_finale) == len(cible), "Nombre de lignes modifié par la fusion des sources"
table_finale.to_parquet(DATA_OUT / "fe_cible_toutes_sources_sbv.parquet", index=False)
print("-> Exporté : fe_cible_toutes_sources_sbv.parquet", table_finale.shape)

# Contrôle : part de valeurs vides par variable (hors cible), de la plus vide à la moins vide
vides = table_finale.drop(columns=CLES_SBV + cols_cible).isna().mean().mul(100).round(1)
print(vides[vides > 0].sort_values(ascending=False).to_string())

-> Exporté : fe_cible_toutes_sources_sbv.parquet (21802, 63)
taux3_vols_vehicules               41.7
taux3_vols_violents                41.7
taux3_violences_intrafamiliales    41.7
taux3_violences_sexuelles          41.7
taux3_destructions                 41.7
taux3_stupefiants                  41.7
taux3_cambriolages                 41.7
taux3_vols_sans_violence           41.7
nb_violences_sexuelles             25.0
nb_violences_intrafamiliales       25.0
nb_vols_vehicules                  25.0
nb_vols_violents                   25.0
nb_destructions                    25.0
nb_stupefiants                     25.0
nb_vols_sans_violence              25.0
nb_cambriolages                    25.0
taux1_vols_sans_violence           25.0
taux1_vols_violents                25.0
part_indicateurs_estimes           25.0
taux1_vols_vehicules               25.0
insee_log                          25.0
insee_pop                          25.0
taux1_destructions                 25.0
taux1_cambriolages 

La seule variable qui nécessite un encodage et qui est en fait cachée dans les autres variables est celle du type de bien. Il est nécessaire de réaliser cette encodage car les marchés sont significativement différents.

Ainsi il faudra doubler les lignes par sous bassin de vie et par année, pour avoir une ligne qui ne concerne que les maisons et une ligne qui ne concerne que les appartements.

Les variables qui vont varier d'une ligne à l'autre : nombre de biens, nombre de ventes, prix médian au m2.
Toutes les autres variables seront identiques car descriptives du territoire dans lequel se trouvent les biens.

In [ ]:
"""
Passage de la table "large" (une ligne par sous-bassin de vie x année,
avec des colonnes séparées pour maisons et appartements) à une table
"longue" (une ligne par sous-bassin x année x type de bien), avec
encodage binaire du type de bien.

Entrée  : fe_cible_toutes_sources_sbv.parquet
Sortie  : fe_cible_toutes_sources_sbv_encodee.parquet
"""

# ---------------------------------------------------------------------------
# 1. Lecture
# ---------------------------------------------------------------------------
# dtype=str sur le code du sous-bassin : sans cela, pandas lit "01004" comme
# l'entier 1004 et le zéro initial est perdu (les codes ont 5 ou 7 caractères).
df = pd.read_parquet(DATA_OUT / "fe_cible_toutes_sources_sbv.parquet")

# ---------------------------------------------------------------------------
# 2. Correspondance entre les colonnes d'origine et les colonnes cibles
# ---------------------------------------------------------------------------
# Trois variables existent en double dans la table d'origine (une version
# "maison", une version "appart"). On les ramène chacune à une seule colonne.
#
# INTERPRÉTATION DE "NOMBRE DE BIENS" ET "NB DE VENTES"
# -----------------------------------------------------
# Le fichier ne contient pas de colonne qui s'appelle littéralement
# "nombre de biens". Voici ce que j'ai déduit en inspectant les données :
#
#   * n_maison / n_appart : nombre de transactions de l'année `annee`
#     (ex. 01004 en 2014 : 368 maisons, 59 appartements).
#     -> c'est la quantité de biens vendus dans l'année. Je l'ai retenue
#        comme "nb_biens".
#
#   * n_ventes_maison_n1 / n_ventes_appart_n1 : ce sont les MÊMES chiffres
#     décalés d'un an (la valeur de 2015 est le n_maison de 2014). Colonne
#     "annee_ventes" = annee - 1 sur toutes les lignes. C'est donc un retard
#     (lag) d'un an, vide pour la première année de chaque sous-bassin.
#     -> retenue comme "nb_ventes_n1".
#-----------------------------------------------------

CORRESPONDANCE = {
    "maison": {
        "nb_biens": "n_maison",
        "nb_ventes_n1": "n_ventes_maison_n1",
        "prix_m2_med": "prix_m2_med_maison",
    },
    "appartement": {
        "nb_biens": "n_appart",
        "nb_ventes_n1": "n_ventes_appart_n1",
        "prix_m2_med": "prix_m2_med_appart",
    },
}

CLES = ["sous_bassin_de_vie", "annee"]

# Colonnes spécifiques à un type de bien (à ne pas recopier telles quelles).
colonnes_specifiques = [
    ancienne
    for mapping in CORRESPONDANCE.values()
    for ancienne in mapping.values()
]

# Toutes les autres colonnes (criminalité, INSEE, équipements...) décrivent le
# territoire et non le type de bien : elles seront dupliquées à l'identique
# sur les deux lignes.
colonnes_communes = [
    c for c in df.columns if c not in colonnes_specifiques and c not in CLES
]

# ---------------------------------------------------------------------------
# 3. Construction d'un bloc par type de bien
# ---------------------------------------------------------------------------
blocs = []
for type_bien, mapping in CORRESPONDANCE.items():
    bloc = df[CLES].copy()

    # Encodage binaire (one-hot) : 1 pour le type de la ligne, 0 pour l'autre.
    bloc["maison"] = int(type_bien == "maison")
    bloc["appartement"] = int(type_bien == "appartement")

    # Variables propres au type de bien, renommées avec un nom commun.
    for nouveau_nom, ancien_nom in mapping.items():
        bloc[nouveau_nom] = df[ancien_nom]

    # Variables du territoire, identiques pour les deux types de bien.
    for c in colonnes_communes:
        bloc[c] = df[c]

    # Ordre d'affichage : maison en premier (0), appartement ensuite (1).
    bloc["_ordre"] = 0 if type_bien == "maison" else 1
    blocs.append(bloc)

# ---------------------------------------------------------------------------
# 4. Empilement et tri
# ---------------------------------------------------------------------------
# Tri par sous-bassin, année, puis type de bien : on obtient pour chaque
# couple sous-bassin x année une ligne maison suivie d'une ligne appartement.
table_longue = (
    pd.concat(blocs)
    .sort_values(CLES + ["_ordre"])
    .drop(columns="_ordre")
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------------
# 5. Contrôles
# ---------------------------------------------------------------------------
# Le nombre de lignes doit avoir exactement doublé.
assert len(table_longue) == 2 * len(df)
# Une et une seule des deux colonnes d'encodage vaut 1 sur chaque ligne.
assert (table_longue["maison"] + table_longue["appartement"] == 1).all()
# Aucune clé (sous-bassin, année, type) en double.
assert not table_longue.duplicated(CLES + ["maison"]).any()



-> Exporté : fe_cible_toutes_sources_sbv_encodee.parquet (43604, 62)


In [39]:
print(table_longue.columns)

Index(['sous_bassin_de_vie', 'annee', 'maison', 'appartement', 'nb_biens',
       'nb_ventes_n1', 'prix_m2_med', 'nb_vols_vehicules', 'nb_vols_violents',
       'nb_vols_sans_violence', 'nb_cambriolages', 'nb_destructions',
       'nb_stupefiants', 'nb_violences_intrafamiliales',
       'nb_violences_sexuelles', 'insee_pop', 'insee_log',
       'part_indicateurs_estimes', 'taux1_vols_vehicules',
       'taux3_vols_vehicules', 'taux1_vols_violents', 'taux3_vols_violents',
       'taux1_vols_sans_violence', 'taux3_vols_sans_violence',
       'taux1_cambriolages', 'taux3_cambriolages', 'taux1_destructions',
       'taux3_destructions', 'taux1_stupefiants', 'taux3_stupefiants',
       'taux1_violences_intrafamiliales', 'taux3_violences_intrafamiliales',
       'taux1_violences_sexuelles', 'taux3_violences_sexuelles', 'annee_faits',
       'taux_vacance_complete', 'part_logements_taux_impute',
       'millesime_lovac', 'nb_equip_commerces', 'nb_equip_sante',
       'nb_equip_services', 'nb_

Avant de procéder à l'export on va en profiter pour :

- générer un nouvel index,

- supprimer les 3 colonnes millesimes qui n'ont qu'une valeur.

In [42]:
# ---------------------------------------------------------------------------
# 6. Génération d'un index annee_sbv_typebien et suppression des colonnes constantes
# ---------------------------------------------------------------------------
# La création de l'index nous permettra de supprimer la colonne du sbv et de ne pas avoir de lignes en doublons/ vérifier qu'il n'y a pas de doublons.
type_bien = table_longue["maison"].map({1: "maison", 0: "appartement"})
table_longue.index = (
    table_longue["annee"].astype(str)
    + "_" + table_longue["sous_bassin_de_vie"]
    + "_" + type_bien
)
table_longue.index.name = "annee_sbv_typebien"

# L'index doit identifier chaque ligne de façon unique.
assert table_longue.index.is_unique

# Les colonnes constantes, par définition, n'apportent pas d'informations.
colonnes_constantes = [c for c in table_longue.columns if table_longue[c].nunique(dropna=True) <= 1]
table_longue = table_longue.drop(columns=colonnes_constantes)

table_longue=table_longue.drop(columns="sous_bassin_de_vie")

In [45]:
# ---------------------------------------------------------------------------
# 7. Export
# ---------------------------------------------------------------------------
# Les valeurs manquantes (prix absent, premier nb_ventes_n1 de chaque
# sous-bassin) restent vides : elles ne sont pas imputées ici.
table_longue.to_parquet(DATA_OUT / "fe_cible_toutes_sources_sbv_encodee.parquet", index=True)
print("-> Exporté : fe_cible_toutes_sources_sbv_encodee.parquet", table_longue.shape)

-> Exporté : fe_cible_toutes_sources_sbv_encodee.parquet (43604, 57)
